**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/du-2026"], check=True)
    os.chdir("/content/encore/entries/du-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Du, Lai, Hu & Ye (2026) on LINK: estimator families

Paper doi:10.64898/2026.08.04.742876; dataset DANDI:001201 version 0.251023.2336 (the cited version;
see `02_link_development.ipynb` for the header, the full claims table and the data check).
This notebook tests estimator-family-link on the 18 development sessions.

| Claim | Status |
|---|---|
| estimator-family-link | tested; reading (b) of the nonlinear context (every setting of Methods 4.6) not run, see below |

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())
T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)

assets = d.list_assets(d.LINK_ID, d.LINK_VERSION)
by_date = collections.defaultdict(list)
for a in assets:
    by_date[d.session_date(a["path"])].append(a)
paired_dates = sorted(k for k, v in by_date.items() if len(v) > 1)
dev_assets = [a for k in paired_dates for a in sorted(by_date[k], key=lambda a: a["path"])]
print(f"{len(assets)} assets in DANDI:{d.LINK_ID} {d.LINK_VERSION}; {len(paired_dates)} dates with two sessions, {len(dev_assets)} development sessions")

def table(results, name, field="gain"):
    return pd.DataFrame([{"date": r["date"], "task": r["task"], field: r[name][field]} for r in results])

In [ ]:
sessions = [d.load_link_session(a["asset_id"], a["path"], signals=("SpikingBandPower", "ThresholdCrossings")) for a in dev_assets]
check = pd.DataFrame([dict(session=s.identifier, trials=len(s.trials), bins=len(s.timestamps),
                           dt_ms=1000 * float(np.median(np.diff(s.timestamps))),
                           sbp_range=f"{s.signals['SpikingBandPower'].min():.3g}..{s.signals['SpikingBandPower'].max():.3g}",
                           tcr_range=f"{s.signals['ThresholdCrossings'].min():.3g}..{s.signals['ThresholdCrossings'].max():.3g}",
                           pos_range=f"{s.position.min():.3g}..{s.position.max():.3g}",
                           vel_range=f"{s.velocity.min():.3g}..{s.velocity.max():.3g}") for s in sessions])
print(check.to_string())
print("tasks:", collections.Counter(s.task for s in sessions), "; units:", sessions[0].units)
print("The full data check of these sessions (duplicates, clock, velocity) is in 02_link_development.ipynb.")
del sessions
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## estimator-family-link

> Separately penalized context and neural blocks and nonlinear context random features retained
> positive LINK task means, with magnitudes that differed between estimator families.

Block penalties: `du2026.block_ridge`, port of `evaluate_block_ridge_audit` (the pair of penalties
over the grid squared on the inner 80/20 split). Nonlinear context: `du2026.rff_context`, port of
`evaluate_rff_context_audit` (D = 256, g chosen from 0.25, 1, 4 with the penalties on the inner split,
five cross-fitting folds), the claims file's reading (a). Reading (b), every setting of Methods 4.6
(D 128/256/512 by three map seeds by g 0.25/1/4, 27 settings), was not run: one setting takes about
18 s per session and context, so 27 settings on 18 sessions and 4 signal-context pairs are about 27
times this notebook's nonlinear-context part (estimated 9 h on four processes). The fully nested
linear means for comparison are those of `02_link_development.ipynb`; they are recomputed here for
the magnitudes.

In [ ]:
jobs = []
for signal, tag in (("SpikingBandPower", "SBP"), ("ThresholdCrossings", "TCR")):
    for context in ("geometry_tensor", "causal_history"):
        build_kw = dict(signal=signal, context_mode=context, seed=20260726)
        jobs.append((f"{tag}_{context}_block", build_kw, "block", {"seed": 20260726}, False))
        jobs.append((f"{tag}_{context}_rffctx", build_kw, "rff_context", {"seed": 20260726}, False))
        jobs.append((f"{tag}_{context}_nested", build_kw, "nested", {"seed": 20260726}, False))
t = time.time()
results = d.run_parallel(d.link_session_jobs, [(a["asset_id"], a["path"], jobs) for a in dev_assets])
print(f"{len(jobs)} fits per session on {len(results)} sessions in {(time.time() - t) / 60:.1f} min")
rows = []
for tag in ("SBP", "TCR"):
    for context in ("geometry_tensor", "causal_history"):
        for family, key in (("fully nested linear", "nested"), ("block penalties", "block"), ("nonlinear context", "rffctx")):
            t_ = table(results, f"{tag}_{context}_{key}")
            rows.append(dict(signal=tag, context=d.LADDER_NAMES[context], family=family, CO=t_[t_.task == "CO"].gain.mean(),
                             RD=t_[t_.task == "RD"].gain.mean(), all18=t_.gain.mean(), positive=int((t_.gain > 0).sum()),
                             min=t_.gain.min(), max=t_.gain.max()))
fam = pd.DataFrame(rows)
print(fam.to_string(float_format=lambda v: f"{v:.5f}"))
alt = fam[fam.family != "fully nested linear"]
print("reading (a), every CO and RD mean positive for both families:", bool((alt[["CO", "RD"]] > 0).all().all()))
print("reading (b), every 18-session mean positive for both families:", bool((alt.all18 > 0).all()))
print("nonlinear-context g chosen:", collections.Counter(r[f"{tag}_{ctx}_rffctx"]["gamma"] for r in results for tag in ("SBP", "TCR") for ctx in ("geometry_tensor", "causal_history")))
print("block penalty pairs chosen:", collections.Counter(r[f"{tag}_{ctx}_block"]["penalties"] for r in results for tag in ("SBP", "TCR") for ctx in ("geometry_tensor", "causal_history")).most_common(6))
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced.** Under both families every CO and RD mean is positive for SBP and TCR, geometry by phase and plus history (reading a), and so is every 18-session mean (reading b). The nonlinear-context means over 18 sessions (0.01733, 0.00663, 0.00949 and 0.00274 for SBP geometry, SBP history, TCR geometry and TCR history) are smaller than the fully nested linear ones (0.02009, 0.00873, 0.01155, 0.00420), and the block-penalty means lie close to the linear ones, so the magnitudes differ between families. Reading (b) of the nonlinear context, every setting of Methods 4.6, was not run.

The block-penalty Ridge above is solved exactly (Cholesky). The release solves it with lsqr (tolerance
1e-6), which is about five times slower and too slow for all 18 sessions within the cell time limit;
the next cell compares the two solvers on one session (2021-10-15 CO, SBP, geometry by phase).

In [ ]:
one = [a for a in dev_assets if d.session_date(a["path"]) == "2021-10-15"]
s = [x for x in (d.load_link_session(a["asset_id"], a["path"], signals=("SpikingBandPower",)) for a in one) if x.task == "CO"][0]
ds = d.build_link_dataset(s, signal="SpikingBandPower", context_mode="geometry_tensor", seed=20260726)
for solver in ("cholesky", "lsqr"):
    t = time.time()
    r = d.block_ridge(ds, seed=20260726, solver=solver)
    print(f"{s.identifier}, SBP geometry by phase, {solver}: gain {r['gain']:.7f}, penalties {r['penalties']}, {time.time() - t:.0f} s")
del s, ds
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

On this session the two solvers choose the same penalties and the gains differ by about 2e-7, far
below the smallest block-penalty task mean (0.00306), so the solver cannot change a sign above. The
other sessions were not run under lsqr.

## Summary

| Claim | Outcome |
|---|---|
| estimator-family-link | reproduced (nonlinear context under the release's setting only) |

The block-penalty Ridge is solved exactly (Cholesky) where the release uses lsqr with tolerance 1e-6;
on one session the two give the same penalties and gains that differ by about 2e-7 (printed above). The nonlinear-context sensitivity follows the release's
`evaluate_rff_context_audit` (D = 256, g chosen on the inner split); the other settings Methods 4.6
lists were not run.